# Augmentations-3: Results / Tests on Data Part 1 - No Aug

In [ ]:
import os
import time
from pathlib import Path

root = Path.cwd()
print(root)
if root.name == "ipynb":
    root = root.parent
    os.chdir(root)
print(root)

import zarr
import tifffile
import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
import ipywidgets as widgets
from scipy.ndimage import rotate

FISBE_DIR = root / "fisbe"

## Helper Functions

### Plotting Segmentation Prediction

In [ ]:
def get_metric_paths(sub_folder:str, cf_stem:str, run:str='0'):
    """Help navigate to biapy results stored in metric folder"""
    # 1. Build the BiaPy results directory: metrics/biapy/{stem}/results/{stem}_{run}/{sub_folder}
    result_path = Path(f"metrics/biapy/{cf_stem}/results/{cf_stem}_{run}/{sub_folder}")
    # 2. Collect all TIFF prediction/instance files in that folder
    image_paths = list(result_path.glob('*.tif'))
    return image_paths


In [ ]:
def mip_biapy_gt_instance(ax, labels, sample_name, z_axis=0, font_siz=6):
    """Plot BiaPy instance MIP onto `ax`. Callable as plot_fn for plot_image_grid."""
    # 1. Max-project instance IDs along Z → 2D label map (Y, X)
    arr = np.asarray(labels)
    mip = arr.max(axis=z_axis)

    # 2. Paint each nonzero instance a random RGB color (skip background=0)
    rgb = np.zeros((*mip.shape, 3), dtype=np.float32)
    for lab in np.unique(mip):
        if lab == 0:
            continue
        color = np.random.randint(72, 255, 3).astype(np.float32)
        rgb[mip == lab] = color
    # 3. Scale to [0, 1] for imshow
    rgb /= 255.0

    # 4. Draw onto the provided axes
    ax.imshow(rgb)
    ax.set_title(f"GT instance MIP {sample_name}", wrap=True)
    ax.title.set_fontsize(font_siz)
    ax.axis("off")


In [ ]:
# Segmentation Prediction Plotting
def plot_image_grid(paths, plot_fn, ncols=4, figsize_cell=(2, 2), dpi=100, **plot_kwargs):
    """Load images from `paths` and draw each with `plot_fn` in a grid of `ncols` columns.

    `plot_fn` signature: plot_fn(ax, data, sample_name, **plot_kwargs)
    """
    # 1. Normalize paths and compute grid layout (rows × ncols)
    paths = [Path(p) for p in paths]
    n = len(paths)
    nrows = max(1, int(np.ceil(n / ncols)))
    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(figsize_cell[0] * ncols, figsize_cell[1] * nrows),
        dpi=dpi,
        squeeze=False,
    )
    axes_flat = axes.ravel()

    # 2. Load each TIFF and hand it to plot_fn (e.g. mip_biapy_gt_instance)
    for i, path in enumerate(paths):
        data = tifffile.imread(path, mode="r")
        plot_fn(axes_flat[i], data, path.stem, **plot_kwargs)

    # 3. Hide unused axes in the last incomplete row
    for j in range(n, len(axes_flat)):
        axes_flat[j].axis("off")
    plt.tight_layout()
    plt.show()
    return fig, axes


### Plotting Raw Channel Predictions

In [ ]:
PCT_LOW, PCT_HIGH, GAMMA = 1.0, 99.5, 0.72
def enhance_display(data: np.ndarray) -> np.ndarray:
    """Shared contrast + gamma; same defaults as web/server/services/volume_pipeline.py."""
    # 1. Work in float32; empty / all-zero → black image
    v = data.astype(np.float32)
    if not np.any(v > 0):
        return np.zeros_like(v)

    # 2. Estimate robust [lo, hi] from percentiles (prefer nonzero voxels)
    sample = v[v > 0] if np.count_nonzero(v) > 256 else v.ravel()
    lo, hi = np.percentile(sample, [PCT_LOW, PCT_HIGH])
    # 3. Fallback to full min/max if percentiles collapse
    if hi <= lo:
        lo, hi = float(v.min()), float(v.max())
    if hi <= lo:
        return np.zeros_like(v)

    # 4. Normalize to [0, 1], then apply gamma (<1 brightens midtones)
    return np.clip((v - lo) / (hi - lo), 0, 1) ** GAMMA


In [ ]:
def resolve_per_image_paths(config_stem, run="0", sub_folder="per_image", n_samples=None):
    """List TIFFs under metrics/biapy/{stem}/results/{stem}_{run}/{sub_folder}."""
    # 1. Locate the BiaPy result subfolder for this config/run
    result_path = Path(f"metrics/biapy/{config_stem}/results/{config_stem}_{run}/{sub_folder}")
    # 2. Sorted glob so sample order is stable across calls
    paths = sorted(result_path.glob("*.tif"))
    if not paths:
        raise FileNotFoundError(f"No .tif files in {result_path}")
    # 3. Optionally truncate to the first n_samples (faster preview)
    if n_samples is not None:
        paths = paths[: max(0, int(n_samples))]
        if not paths:
            raise ValueError("n_samples resolved to an empty path list")
    return paths


def resolve_prediction_channels(n_p_all, p=None, channel_names=None):
    """Normalize `p` to indices and build column labels. Returns (p_idx, labels)."""
    # 1. Resolve which prediction-channel indices to show (None → all)
    if p is None:
        p_idx = list(range(n_p_all))
    else:
        p_idx = [int(p)] if isinstance(p, (int, np.integer)) else [int(x) for x in p]
        bad = [j for j in p_idx if j < 0 or j >= n_p_all]
        if bad:
            raise ValueError(f"p out of range for P={n_p_all}: {bad}")

    # 2. Build column labels: defaults p0.., or subset/full channel_names list
    if channel_names is None:
        labels = [f"p{j}" for j in p_idx]
    elif len(channel_names) == n_p_all:
        labels = [channel_names[j] for j in p_idx]
    elif len(channel_names) == len(p_idx):
        labels = list(channel_names)
    else:
        raise ValueError(
            f"channel_names length {len(channel_names)} must be P={n_p_all} or len(p)={len(p_idx)}"
        )
    return p_idx, labels


def prediction_channel_mip(vol_zpyx, channel, threshold=None):
    """Max-project one prediction channel of (Z, P, Y, X) → (Y, X); optional Fiji-style cutoff."""
    # 1. Take channel slice over Z and max-project → (Y, X)
    mip = np.asarray(vol_zpyx[:, channel]).max(axis=0)
    # 2. Optional cutoff: zero voxels below threshold (keep supra-threshold intensities)
    if threshold is not None:
        mip = np.where(mip >= threshold, mip, 0)
    return mip


def auto_prediction_cmap(mip, cmap=None):
    """Binary (≤2 unique values) → gray; continuous → viridis. `cmap` overrides."""
    # Explicit override wins; else gray for binary masks, viridis for continuous scores
    if cmap is not None:
        return cmap
    return "gray" if np.unique(mip).size <= 2 else "viridis"


def draw_prediction_subplot(ax, mip, *, title=None, ylabel=None, font_siz=6, cmap="viridis"):
    """imshow display-enhanced MIP on `ax` with optional title/ylabel."""
    # 1. Contrast/gamma enhance, then imshow
    ax.imshow(enhance_display(mip), cmap=cmap)
    # 2. Optional labels (top-row titles / left-column sample names)
    if title is not None:
        ax.set_title(title, fontsize=font_siz)
    if ylabel is not None:
        ax.set_ylabel(ylabel, fontsize=font_siz)
    ax.set_xticks([])
    ax.set_yticks([])

# Raw Channel Predictions
def plot_per_image_predictions(
    config_stem: str,
    run: str = "0",
    *,
    sub_folder: str = "per_image",
    n_samples=None,
    p=None,
    threshold=None,
    figsize_cell=(3, 3),
    dpi=100,
    channel_names=None,
    font_siz=6,
    cmap=None,
):
    """Plot prediction-channel MIPs for TIFFs in a BiaPy result folder.

    Reads ``metrics/biapy/{config_stem}/results/{config_stem}_{run}/{sub_folder}``.
    Each image is ``(Z, P, Y, X)``; the figure has one row per sample and one column
    per selected prediction channel (max-intensity projection over Z).

    Colormap is chosen per subplot: ``gray`` when the MIP has ≤2 unique values,
    else ``viridis``. Pass ``cmap`` to override for all subplots.

    Parameters
    ----------
    n_samples : int or None
        If set, only load/plot the first ``n_samples`` TIFFs (faster).
    p : int, sequence of int, or None
        Prediction channel index/indices to show. ``None`` = all channels.
    threshold : float or None
        If set, values below ``threshold`` on the MIP are zeroed (Fiji-style cutoff).
    """
    # 1. Resolve TIFF paths under the BiaPy per_image (or similar) folder
    #    Layout is (Z, P, Y, X): Z depth, P prediction channels, then Y/X.
    paths = resolve_per_image_paths(config_stem, run, sub_folder, n_samples)

    # 2. Peek P from the first volume only (memmap) so we can validate `p` before the loop
    first = tifffile.imread(paths[0], mode="r")
    if first.ndim != 4:
        raise ValueError(f"Expected (Z, P, Y, X), got shape {first.shape} for {paths[0].name}")
    n_p_all = int(first.shape[1])
    del first

    # 3. Normalize channel selection → column indices + labels; allocate figure
    p_idx, labels = resolve_prediction_channels(n_p_all, p, channel_names)
    n, n_cols = len(paths), len(p_idx)
    fig, axes = plt.subplots(
        n,
        n_cols,
        figsize=(figsize_cell[0] * n_cols, figsize_cell[1] * n),
        dpi=dpi,
        squeeze=False,
    )

    # 4. Per sample × channel: MIP → enhance → draw (row=sample, col=channel)
    for i, path in enumerate(paths):
        # Memmap + project only selected channels so we do not materialize full volumes.
        data = tifffile.imread(path, mode="r")  # (Z, P, Y, X)
        for col, j in enumerate(p_idx):
            mip = prediction_channel_mip(data, j, threshold=threshold)
            draw_prediction_subplot(
                axes[i, col],
                mip,
                title=labels[col] if i == 0 else None,
                ylabel=path.stem if col == 0 else None,
                font_siz=font_siz,
                cmap=auto_prediction_cmap(mip, cmap),
            )
        del data

    # 5. Title + layout
    thresh_txt = f", thr≥{threshold}" if threshold is not None else ""
    fig.suptitle(f"{config_stem}_{run} / {sub_folder}{thresh_txt}", fontsize=font_siz + 2)
    plt.tight_layout()
    plt.show()
    return fig, axes


### Plotting Ground Truths Labels

In [ ]:
def resolve_gt_channel_paths(gt_path, n_samples=None):
    """List TIFFs under a BiaPy instance-channel GT folder (label_F.…_Dn.…)."""
    # 1. Point at the GT label folder (e.g. fisbe/.../train/label_F.…_Dn.…)
    result_path = Path(gt_path)
    # 2. Sorted TIFF list; optionally truncate for a quick preview
    paths = sorted(result_path.glob("*.tif"))
    if not paths:
        raise FileNotFoundError(f"No .tif files in {result_path}")
    if n_samples is not None:
        paths = paths[: max(0, int(n_samples))]
        if not paths:
            raise ValueError("n_samples resolved to an empty path list")
    return paths

# Ground Truths Labels
def plot_gt_instance_channels(
    gt_path,
    *,
    n_samples=None,
    p=None,
    threshold=None,
    figsize_cell=(3, 3),
    dpi=100,
    channel_names=None,
    font_siz=6,
    cmap=None,
):
    """Plot BiaPy instance-channel GT MIPs (F/C/Db/Dn targets the network learns).

    Reads TIFFs from ``gt_path`` (e.g. ``fisbe/.../train/label_F.…_Dn.…``).
    Each image is ``(Z, P, Y, X)`` — same layout as ``per_image`` predictions.
    Reuses the prediction MIP / cmap / draw helpers.

    Parameters
    ----------
    gt_path : str or Path
        Directory of multi-channel GT TIFFs.
    n_samples, p, threshold, channel_names, cmap
        Same meaning as ``plot_per_image_predictions``.
        If ``channel_names`` is None and ``P==4``, defaults to ``['F','C','Db','Dn']``.
    """
    # 1. List GT TIFFs — same (Z, P, Y, X) layout as predictions
    #    (F/C often binary; Db/Dn continuous distance-like targets)
    paths = resolve_gt_channel_paths(gt_path, n_samples)

    # 2. Infer channel count P from the first volume
    first = tifffile.imread(paths[0], mode="r")
    if first.ndim != 4:
        raise ValueError(f"Expected (Z, P, Y, X), got shape {first.shape} for {paths[0].name}")
    n_p_all = int(first.shape[1])
    del first

    # 3. Default F/C/Db/Dn names when P==4 and caller did not supply labels
    if channel_names is None and n_p_all == 4:
        channel_names = ["F", "C", "Db", "Dn"]

    # 4. Resolve channels + allocate figure (rows=samples, cols=channels)
    p_idx, labels = resolve_prediction_channels(n_p_all, p, channel_names)
    n, n_cols = len(paths), len(p_idx)
    fig, axes = plt.subplots(
        n,
        n_cols,
        figsize=(figsize_cell[0] * n_cols, figsize_cell[1] * n),
        dpi=dpi,
        squeeze=False,
    )

    # 5. Per sample × channel: MIP → enhance → draw (reuse prediction helpers)
    for i, path in enumerate(paths):
        data = tifffile.imread(path, mode="r")  # (Z, P, Y, X)
        for col, j in enumerate(p_idx):
            mip = prediction_channel_mip(data, j, threshold=threshold)
            draw_prediction_subplot(
                axes[i, col],
                mip,
                title=labels[col] if i == 0 else None,
                ylabel=path.stem if col == 0 else None,
                font_siz=font_siz,
                cmap=auto_prediction_cmap(mip, cmap),
            )
        del data

    # 6. Title + layout
    thresh_txt = f", thr≥{threshold}" if threshold is not None else ""
    fig.suptitle(f"GT channels: {Path(gt_path).name}{thresh_txt}", fontsize=font_siz + 2)
    plt.tight_layout()
    plt.show()
    return fig, axes


### Stats

In [ ]:
def per_channel_stats(data: np.ndarray, axis: int = 0) -> pd.DataFrame:
    """Basic stats of values across channels (axis 0 for CZYX).

    Expects shape like (C, Z, Y, X), e.g. (3, 390, 680, 680) uint16.
    """
    # 1. Flatten each channel along the chosen axis (default 0 = C in CZYX)
    data = np.asarray(data)
    n_ch = data.shape[axis]
    rows = []
    for c in range(n_ch):
        ch = np.take(data, c, axis=axis).ravel()
        nonzero = ch[ch > 0]
        # 2. Record global + nonzero-only intensity stats for this channel
        rows.append({
            "channel": c,
            "dtype": str(data.dtype),
            "n": ch.size,
            "n_nonzero": int(nonzero.size),
            "min": int(ch.min()) if ch.size else np.nan,
            "max": int(ch.max()) if ch.size else np.nan,
            "mean": float(ch.mean()) if ch.size else np.nan,
            "std": float(ch.std()) if ch.size else np.nan,
            "median": float(np.median(ch)) if ch.size else np.nan,
            "p1": float(np.percentile(ch, 1)) if ch.size else np.nan,
            "p99": float(np.percentile(ch, 99)) if ch.size else np.nan,
            "mean_nonzero": float(nonzero.mean()) if nonzero.size else 0.0,
            "median_nonzero": float(np.median(nonzero)) if nonzero.size else 0.0,
        })
    # 3. One row per channel, indexed by channel id
    return pd.DataFrame(rows).set_index("channel")


def per_channel_histogram(
    data: np.ndarray,
    sample_name: str,
    axis: int = 0,
    bins: int = 256,
    range_=None,
    skip_zeros: bool = True,
    log_y: bool = True,
    figsize=(12, 3.5),
):
    """Basic histogram of value distribution across channels (axis 0 for CZYX).

    Plots one subplot per channel. Returns (fig, axes, counts).
    """
    # 1. Shared value range across channels (default: data min → max+1)
    data = np.asarray(data)
    n_ch = data.shape[axis]
    if range_ is None:
        range_ = (int(data.min()), int(data.max()) + 1)

    # 2. One subplot per channel, shared y for easy comparison
    fig, axes = plt.subplots(1, n_ch, figsize=figsize, sharey=True)
    if n_ch == 1:
        axes = [axes]

    # 3. Histogram each channel (optionally drop zeros — common for sparse volumes)
    counts = []
    for c, ax in enumerate(axes):
        ch = np.take(data, c, axis=axis).ravel()
        if skip_zeros:
            ch = ch[ch > 0]
        hist, edges = np.histogram(ch, bins=bins, range=range_)
        counts.append(hist)
        centers = (edges[:-1] + edges[1:]) / 2
        ax.bar(centers, hist, width=np.diff(edges), align="center", alpha=0.8)
        ax.set_title(f"channel {c}")
        ax.set_xlabel("value")
        if log_y:
            ax.set_yscale("log")
        ax.set_xlim(range_)

    axes[0].set_ylabel("count" + (" (log)" if log_y else ""))
    fig.suptitle(f"Per-channel value histogram {sample_name}", y=1.02)
    fig.tight_layout()
    return fig, axes, counts


## Viewing Results

In [ ]:
CONFIG_STEM = 'biapy-v1-channel-scale'
SUB_FOLDER = 'per_image_post_processing'
RUN = '0'
result_path = Path(f"metrics/biapy/{CONFIG_STEM}/results/{CONFIG_STEM}_{RUN}/{SUB_FOLDER}")
image_paths = list(result_path.glob('*.tif'))
example_sample = tifffile.imread(image_paths[0], mode='r')
print(f"Image sample shape: {example_sample.shape}")
# print(f"Num Unique Values: {len(np.unique(example_sample))}")


def get_gt_paths(full_path:str):
    """Full path is mean to be .../label_F.erosion..."""
    # 1. Point at a BiaPy GT label directory (label_F.erosion… / etc.)
    result_path = Path(full_path)
    # 2. Collect all TIFF volumes in that folder
    image_paths = list(result_path.glob('*.tif'))
    return image_paths


# SUB_FOLDER
# per_image (z, 4, x, y)
# per_image_instances (z, x, y)
# per_image_post_processing (z, x, y)
# train_FCDbDn_instance_channels (z, x, y)


### Segmentation Prediction

In [ ]:
plot_image_grid(get_metric_paths('per_image_instances', 'biapy-v1-no-aug'), mip_biapy_gt_instance, ncols=4, figsize_cell=(4, 4), font_siz=6)
plot_image_grid(get_metric_paths('per_image_instances', 'biapy-v2-no-aug'), mip_biapy_gt_instance, ncols=4, figsize_cell=(4, 4), font_siz=6)
plot_image_grid(get_metric_paths('per_image_instances', 'biapy-v3-no-aug'), mip_biapy_gt_instance, ncols=4, figsize_cell=(4, 4), font_siz=6)


In [ ]:
plot_image_grid(get_metric_paths('per_image_instances', 'biapy-v4-no-aug-seunet'), mip_biapy_gt_instance, ncols=4, figsize_cell=(4, 4), font_siz=6)
plot_image_grid(get_metric_paths('per_image_instances', 'biapy-v4-no-aug-unetr'), mip_biapy_gt_instance, ncols=4, figsize_cell=(4, 4), font_siz=6)


### Ground Truths Labels
See the ground truth lables for different channesl. (e.g. F, C, ...)

#### DATA_CHANNELS: ['F', 'C', 'Db', 'Dn']

In [ ]:
gt_path = "fisbe/biapy-no-aug/train/label_F.erosion-0.dilation-0_C.mode-thick_Db.val_type-norm.act-sigmoid.mask_values-True_Dn.closing_size-3.norm-True.mask_values-True.decline_power-3"

# gt_path = "fisbe/biapy-no-aug/train/label_F.erosion-0.dilation-0_T.thickness-2"
# gt_path = "fisbe/biapy-no-aug/train/label_F.erosion-0.dilation-0_P.type-skeleton.dilation-1.erosion-0"
# gt_path = "fisbe/biapy-no-aug/test/label_F.erosion-0.dilation-0_T.thickness-2_P.type-skeleton.dilation-1.erosion-0"
figsize_cell=(4,4)

get_gt_paths(gt_path)
plot_gt_instance_channels(gt_path, n_samples=9, figsize_cell=figsize_cell)


#### DATA_CHANNELS: ['F', 'T', 'P']

In [ ]:
gt_path = "fisbe/biapy-no-aug/train/label_F.erosion-0.dilation-0_T.thickness-2_P.type-skeleton.dilation-1.erosion-0"
figsize_cell=(4,4)
get_gt_paths(gt_path)
plot_gt_instance_channels(gt_path, n_samples=9, figsize_cell=figsize_cell)


### Raw Channel Predictions
Hopefully helps with deciding which channesl, seed channels, growth channels, thresholds to choose

In [ ]:
CFG_STEM = 'biapy-v1-no-aug'
figsize_cell = (6, 6)
n_samples = 3
plot_per_image_predictions(CFG_STEM, n_samples=n_samples, threshold=0.0, figsize_cell=figsize_cell)
plot_per_image_predictions(CFG_STEM, n_samples=n_samples, threshold=0.25, figsize_cell=figsize_cell)
plot_per_image_predictions(CFG_STEM, n_samples=n_samples, threshold=0.5, figsize_cell=figsize_cell)
plot_per_image_predictions(CFG_STEM, n_samples=n_samples, threshold=0.75, figsize_cell=figsize_cell)

In [ ]:
CFG_STEM = 'biapy-v2-no-aug'
figsize_cell = (4, 4)
n_samples = 3
plot_per_image_predictions(CFG_STEM, n_samples=n_samples, threshold=0.0, figsize_cell=figsize_cell)
plot_per_image_predictions(CFG_STEM, n_samples=n_samples, threshold=0.25, figsize_cell=figsize_cell)
plot_per_image_predictions(CFG_STEM, n_samples=n_samples, threshold=0.5, figsize_cell=figsize_cell)
plot_per_image_predictions(CFG_STEM, n_samples=n_samples, threshold=0.75, figsize_cell=figsize_cell)

#### Different Model Arch

In [ ]:
CFG_STEM = 'biapy-v4-no-aug-seunet'
figsize_cell = (4, 4)
n_samples = 3
plot_per_image_predictions(CFG_STEM, n_samples=n_samples, threshold=0.0, figsize_cell=figsize_cell)
plot_per_image_predictions(CFG_STEM, n_samples=n_samples, threshold=0.25, figsize_cell=figsize_cell)
plot_per_image_predictions(CFG_STEM, n_samples=n_samples, threshold=0.5, figsize_cell=figsize_cell)
plot_per_image_predictions(CFG_STEM, n_samples=n_samples, threshold=0.75, figsize_cell=figsize_cell)

In [ ]:
CFG_STEM = 'biapy-v4-no-aug-unetr'
figsize_cell = (4, 4)
n_samples = 3
plot_per_image_predictions(CFG_STEM, n_samples=n_samples, threshold=0.0, figsize_cell=figsize_cell)
plot_per_image_predictions(CFG_STEM, n_samples=n_samples, threshold=0.25, figsize_cell=figsize_cell)
plot_per_image_predictions(CFG_STEM, n_samples=n_samples, threshold=0.5, figsize_cell=figsize_cell)
plot_per_image_predictions(CFG_STEM, n_samples=n_samples, threshold=0.75, figsize_cell=figsize_cell)